In [5]:
### Data ingestion


In [6]:
from langchain_core.documents import Document


In [7]:
import os 
os.makedirs('../data/text_files',exist_ok=True)


In [8]:
sample_text={
    "../data/text_files/sample.text": """LangChain is a powerful framework for building applications with language models.
      It provides a set of tools and abstractions that make it easier to work with language models, 
      allowing developers to create complex applications that can understand and generate human-like text. With LangChain,
        you can build chatbots, virtual assistants, and other applications that require natural language understanding and generation."""
}

for file_path, content in sample_text.items():
    with open(file_path,'w',encoding='utf-8') as f:
        f.write(content)

print("sample text files created successfully ")

sample text files created successfully 


In [9]:
### text loader 
#from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader


In [10]:
loader=TextLoader("../data/text_files/sample.text")

In [11]:
documents=loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/sample.text'}, page_content='LangChain is a powerful framework for building applications with language models.\n      It provides a set of tools and abstractions that make it easier to work with language models, \n      allowing developers to create complex applications that can understand and generate human-like text. With LangChain,\n        you can build chatbots, virtual assistants, and other applications that require natural language understanding and generation.')]


In [12]:
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    '../data/text_files',
    glob='**/*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False
)

In [13]:
print(dir_loader.load())

[Document(metadata={'source': '..\\data\\text_files\\sample.txt'}, page_content='LangChain is a powerful framework for building applications with language models.\n      It provides a set of tools and abstractions that make it easier to work with language models, \n      allowing developers to create complex applications that can understand and generate human-like text. With LangChain,\n        you can build chatbots, virtual assistants, and other applications that require natural language understanding and generation.')]


In [14]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
pdf_loader=DirectoryLoader(
    '../data/pdf_files',
    glob='**/*.pdf',
    loader_cls=PyMuPDFLoader,
    show_progress=False

    )
pdf_documents=pdf_loader.load()
pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-28T17:39:30+00:00', 'source': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'file_path': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-03-28T17:39:30+00:00', 'trapped': '', 'modDate': "D:20260328173930+00'00'", 'creationDate': "D:20260328173930+00'00'", 'page': 0}, page_content='Machine Learning - Detailed Overview\nOverview\nMachine Learning (ML) is a branch of Artificial Intelligence that focuses on enabling systems to\nlearn from data.\nInstead of being explicitly programmed, ML models identify patterns and make decisions with\nminimal human intervention.\nIt is widely used in modern applications such as recommendation systems, fraud detection, and\nhealthcare analytics.\n1\nSubset of AI\n2\nData-driven approach\n3\nImpro

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_chunks(documents,chunk_size=100,chunk_overlap=20):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_chunks)} chunks")
    
    # Show what a chunk looks like
    if split_chunks:
        print(f"\nExample chunk:")
        print(f"Content: {split_chunks[0].page_content[:200]}...")
        print(f"Metadata: {split_chunks[0].metadata}")
    
    return split_chunks

In [16]:
chunks=split_chunks(pdf_documents)

Split 4 documents into 31 chunks

Example chunk:
Content: Machine Learning - Detailed Overview
Overview...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-28T17:39:30+00:00', 'source': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'file_path': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-03-28T17:39:30+00:00', 'trapped': '', 'modDate': "D:20260328173930+00'00'", 'creationDate': "D:20260328173930+00'00'", 'page': 0}


### embeddings and VectorStore DB

In [17]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb # vector database
import faiss #vector similarity search library
from chromadb.config import Settings
import uuid # used for to assign the index into vectorebase
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [18]:
class EmbeddingManager:
    def __init__ (self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name  =model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print("model is loading... ",self.model_name)
            self.model=SentenceTransformer(self.model_name)
            print("model is loaded successfully. embedding dimension:",self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print("Error occurred while loading the model: ", e)
            raise 

    def get_embedding(self,texts:List[str])->np.ndarray:
        if self.model is None:
            raise ValueError("Model is not loaded.")
        try:
            embeddings=self.model.encode(texts,show_progress_bar=True)
            return np.array(embeddings)
        except Exception as e:
            print("Error occurred while generating embeddings: ", e)
            raise

    #def get_sentence_embedding_dimension(self)->int:
    #    if not self.model:
     #       raise ValueError("Model is not loaded")
      #  return self.get_sentence_embedding_dimension()


embedding_manager=EmbeddingManager()
embedding_manager








model is loading...  all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8486.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model is loaded successfully. embedding dimension: 384


### vector store 


In [19]:
class VectorStore:
    def __init__(self,collection_name:str='pdf_documents',persist_directory:str='../data/vector_store'):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    
    def _initialize_store(self):
        try: # create persistent chromadb client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            
            #create or get collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'description':'collection of pdf document embeddings'}

            )
            print(f'Vector store initialized successfully. Collection name: {self.collection_name}')
            print(f'existing documents  in collection: {self.collection.count()}')
        except Exception as e:
            print("Error occurred while initializing vector store: ", e)
            raise   


    def add_documents(self,documents:List[any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same.")
        print(f"adding {len(documents)} documents to vector store...")

        #perpare data for chromadb
        ids=[]
        metadatas=[]
        documents_texts=[]
        embeddings_list=[]

        for i,(doc,embeddings) in enumerate(zip(documents,embeddings)):
            doc_id=f'doc_{uuid.uuid4().hex[:8]} {i}'
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            documents_texts.append(doc.page_content)
            embeddings_list.append(embeddings.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list

            )
            print(f'successfully added {len(documents)} documents to vector store. Total documents in collection: {self.collection.count()}')
        except Exception as e:
            print("Error occurred while adding documents to vector store: ", e)
            raise

vectorstore=VectorStore()
vectorstore







Vector store initialized successfully. Collection name: pdf_documents
existing documents  in collection: 88


In [20]:
split_chunks(pdf_documents)

Split 4 documents into 31 chunks

Example chunk:
Content: Machine Learning - Detailed Overview
Overview...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-28T17:39:30+00:00', 'source': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'file_path': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-03-28T17:39:30+00:00', 'trapped': '', 'modDate': "D:20260328173930+00'00'", 'creationDate': "D:20260328173930+00'00'", 'page': 0}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-28T17:39:30+00:00', 'source': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'file_path': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-03-28T17:39:30+00:00', 'trapped': '', 'modDate': "D:20260328173930+00'00'", 'creationDate': "D:20260328173930+00'00'", 'page': 0}, page_content='Machine Learning - Detailed Overview\nOverview'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-28T17:39:30+00:00', 'source': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'file_path': '..\\data\\pdf_files\\ml_detailed_1.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-03-

In [21]:
# convert text to embeddings
texts=[doc.page_content for doc in chunks]

embedding=embedding_manager.get_embedding(texts)

vectorstore.add_documents(chunks,embedding)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.48it/s]

adding 31 documents to vector store...
successfully added 31 documents to vector store. Total documents in collection: 119


###  retriver pipeline from vectorstore

In [48]:
class RAGRetriver:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threeshold: float = 0.5) -> List[Dict[str, Any]]:
        print(f"Retrieving relevant documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threeshold}")

        query_embedding = self.embedding_manager.get_embedding([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc, meta, dist, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                    similarity_score = 1 - dist
                    if similarity_score >= score_threeshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": doc,
                                "metadata": meta,
                                "similarity_score": similarity_score,
                                "distance": dist,
                                "rank": i + 1
                            }
                        )

                if retrieved_docs:
                    print(f"Retrieved {len(retrieved_docs)} relevant documents.")
                else:
                    print("No relevant documents found for the query.")

            return retrieved_docs
        except Exception as e:
            print("Error occurred while retrieving documents:", e)
            return []

retriever = RAGRetriver(vectorstore, embedding_manager)
retriever

retriever.retrieve("what is Machine Learning?")

Retrieving relevant documents for query: 'what is Machine Learning?'
Top K: 5, Score Threshold: 0.5


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.59it/s]

Retrieved 5 relevant documents.


[{'id': 'doc_5844562f 1',
  'content': 'Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.',
  'metadata': {'source': '..\\data\\pdf_files\\ml_pdf_1.pdf',
   'trapped': '',
   'format': 'PDF 1.4',
   'subject': '(unspecified)',
   'producer': 'ReportLab PDF Library - (opensource)',
   'file_path': '..\\data\\pdf_files\\ml_pdf_1.pdf',
   'total_pages': 1,
   'creationDate': "D:20260328160426+00'00'",
   'content_length': 96,
   'keywords': '',
   'modDate': "D:20260328160426+00'00'",
   'creationdate': '2026-03-28T16:04:26+00:00',
   'creator': '(unspecified)',
   'title': '(anonymous)',
   'author': '(anonymous)',
   'moddate': '2026-03-28T16:04:26+00:00',
   'page': 0,
   'doc_index': 1},
  'similarity_score': 0.6622486412525177,
  'distance': 0.3377513587474823,
  'rank': 1},
 {'id': 'doc_06229293 1',
  'content': 'Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.',
  'metadata': {'content_le

### integrate vectorDB  context pipeline with llm output 

In [49]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# initialize the Groq LLM with a currently supported model
grok_api_key = os.getenv("GROQ_API_KEY")
groq_llm = ChatGroq(
    api_key=grok_api_key,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024,
 )

# simple RAG function: retrieve context and generate response
def simple_rag(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "sorry i could not find any relevant information to answer your question."

    prompt = f"Use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt)
    return response.content

In [50]:
answer=simple_rag("what is supervised Learning",retriever,groq_llm )
print(answer)

Retrieving relevant documents for query: 'what is supervised Learning'
Top K: 3, Score Threshold: 0.5


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.98it/s]

Retrieved 2 relevant documents.


Supervised learning is a type of machine learning in which a model is trained on labeled data, using input–output pairs to learn a mapping from inputs to desired outputs.


### improving RAG pipeline

In [71]:
def rag_advanced(query, retriever, llm, top_k=3, min_score=0.35, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threeshold=min_score)

    # Fallback: if strict threshold removes all matches, retry without threshold filtering.
    if not results and min_score > 0:
        results = retriever.retrieve(query, top_k=top_k, score_threeshold=0.0)

    if not results:
        output = {
            "answer": "I could not find a reliable answer in the indexed documents.",
            "sources": [],
            "confidence": 0.0,
        }
        if return_context:
            output["context"] = ""
        return output

    # Remove duplicate chunks that can appear after repeated indexing runs.
    deduped = {}
    for doc in results:
        key = (
            doc["metadata"].get("source", "unknown"),
            doc["metadata"].get("page", "unknown"),
            doc["content"].strip(),
        )
        if key not in deduped or doc["similarity_score"] > deduped[key]["similarity_score"]:
            deduped[key] = doc
    results = sorted(deduped.values(), key=lambda d: d["similarity_score"], reverse=True)[:top_k]

    context = "\n\n".join([doc["content"] for doc in results])
    sources = [
        {
            "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:200] + "...",
        }
        for doc in results
    ]

    confidence = max([doc["similarity_score"] for doc in results])
    prompt = f"""You are a helpful assistant for question answering over documents.

Rules:
- Use ONLY the provided context.
- Give a direct answer in 2-4 sentences, in simple English.
- Do not repeat the question.
- If context is insufficient, say: 'The documents do not provide enough information.'

Context:
{context}

Question: {query}
Answer:"""
    response = llm.invoke(prompt)

    output = {
        "answer": response.content.strip(),
        "sources": sources,
        "confidence": confidence,
    }
    if return_context:
        output["context"] = context
    return output

result = rag_advanced("what is PCA ", retriever, groq_llm, top_k=3, min_score=0.35, return_context=True)
print("Answer:\n", result["answer"])

print("\nConfidence:", round(result["confidence"], 3))

print("\nTop sources:")

for s in result["sources"]:
    print(f"- {s['source']} (page {s['page']}), score={s['score']:.3f}")

Retrieving relevant documents for query: 'what is PCA '
Top K: 3, Score Threshold: 0.35


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.22it/s]


No relevant documents found for the query.
Retrieving relevant documents for query: 'what is PCA '
Top K: 3, Score Threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.95it/s]


Retrieved 3 relevant documents.
Answer:
 PCA stands for Principal Component Analysis. It is a technique used to analyze data in many fields.

Confidence: 0.006

Top sources:
- ..\data\pdf_files\ml_detailed_3.pdf (page 0), score=0.006


In [78]:
#--- advanced Rag pipeline : streaming , citations, history , summarization
from typing import List, Dict, Any  
import time 

class advancedRAGPipeline:
    def __init__(self,retriever,llm):
        self.retriever=retriever
        self.llm=llm
        self.history=[] 
    def query(self, question:str,top_k:int=5,min_score:float=0.2,stream:bool=False,summarize:bool=False)->Dict[str,Any]:
        results=self.retriever.retrieve(question,top_k=top_k,score_threeshold=min_score)
        if not results:
            answer='no relevent context found'
            sources=[]
            context=''
        else:
            context="\n\n".join([doc["content"] for doc in results])
            sources=[
                {
                    "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:200]+"..."
                }
                for doc in results
            ]
            prompt=f""" use the following context to answer the question concisely .\n {context}\n\nQuestion:{question}\nAnswer:"""
            if stream:
                print("streaming answer",flush=True)
                for i in range(0,len(prompt),80):
                    print(prompt[i:i+80],end='',flush=True)
                    time.sleep(0.5)
                print()
            response=self.llm.invoke(prompt.format(context=context,question=question))
            answer=response.content
        citations=[f"[{i+1}]{src['source']} (page {src['page']})" for i,src in enumerate(sources)]
        answer_with_citations=answer+"\n\nSources:\n"+ "\n".join(citations) if citations else answer    
        summary=None
        if summarize and context:
            summary_prompt=f"""Summarize the following context in 2-3 sentences:\n\n{context}"""
            summary_response=self.llm.invoke(summary_prompt)
            summary=summary_response.content
        self.history.append({
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary
        })
        return {
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }
adv_rag=advancedRAGPipeline(retriever,groq_llm)
result=adv_rag.query("what is unsupervised learning?",top_k=3,min_score=0.1,stream=True,summarize=True)
print("Answer with citations:\n",result["answer"])  
print("\nSummary:\n",result["summary"])
print("\nConversation history:",result["history"]  [-1])




Retrieving relevant documents for query: 'what is unsupervised learning?'
Top K: 3, Score Threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.38it/s]

Retrieved 3 relevant documents.
streaming answer
 use the following context to answer the question concisely .
 Unsupervised Lear

ning - Detailed
Concept
Unsupervised learning deals with unlabeled data.

Unsupervised Learning - Detailed
Concept
Unsupervised learning deals with unlabeled data.

Unsupervised Learning - Detailed
Concept
Unsupervised learning deals with unlabeled data.

Question:what is unsupervised learning?
Answer:
Answer with citations:
 Unsupervised learning is a type of machine learning that works with unlabeled data, discovering patterns, structures, or groupings without predefined target outputs.

Sources:
[1]..\data\pdf_files\ml_detailed_3.pdf (page 0)
[2]..\data\pdf_files\ml_detailed_3.pdf (page 0)
[3]..\data\pdf_files\ml_detailed_3.pdf (page 0)

Summary:
 Unsupervised learning is a type of machine learning that works with unlabeled data. It focuses on discovering hidden patterns or structures in the data without any predefined labels.

Conversation history: {'question': 'what is unsupervised learning?', 'answer': 'Unsupervised learning is a type of machine learning that works with unlabeled